# WasteLens - Iteration 12 (Colab GPU)


Protocol: docs/rejection_experiment/iteration12_distillation_protocol.md (registered pre-training, commit 3b88e70). Run every code cell top to bottom; every heavy cell STOPS the run on any non-zero exit - nothing may be skipped, reordered, or re-run after completion.


## S0 - environment


In [ ]:
!nvidia-smi


In [30]:
import sys, platform
import tensorflow as tf
print('Python:', sys.version.split()[0], '|', platform.platform())
print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus if gpus else 'NONE - enable a GPU runtime')
assert gpus, 'STOP: no GPU visible'
import numpy, sklearn, PIL
print('numpy:', numpy.__version__,
      '| sklearn:', sklearn.__version__,
      '| pillow:', PIL.__version__)


Python: 3.13.15 | Linux-6.6.122+-x86_64-with-glibc2.39
TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
numpy: 2.1.3 | sklearn: 1.6.1 | pillow: 11.3.0


## S1 - pin the repo: checkout the registered commit


In [31]:
import os, subprocess, sys
from pathlib import Path

PINNED = 'cf2787b0746506256771a54c3c69feaf6ce3618e'
REPO = '/content/WasteLens'

if not Path(REPO, 'src', 'train_distill_adaptation.py').exists():
    subprocess.run(['git', 'clone',
                    'https://github.com/Anik00223/WasteLens.git',
                    REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', '--force', PINNED], check=True)

head = subprocess.run(['git', 'rev-parse', 'HEAD'],
                      capture_output=True, text=True,
                      check=True).stdout.strip()
origin = subprocess.run(['git', 'rev-parse', 'origin/main'],
                        capture_output=True, text=True,
                        check=True).stdout.strip()
print('HEAD        :', head)
print('PINNED      :', PINNED)
print('origin/main :', origin, '(informational - the pin wins)')
assert head == PINNED, ('STOP: HEAD ' + head + ' != PINNED '
                        + PINNED)
MANIFESTS = ['docs/rejection_experiment/adaptation_set_manifest.json',
             'docs/rejection_experiment/fresh_set_manifest.json']
print('repo pinned OK')


HEAD        : cf2787b0746506256771a54c3c69feaf6ce3618e
PINNED      : cf2787b0746506256771a54c3c69feaf6ce3618e
origin/main : 63c67d3dd653b93e0c10a502e13d7ce6eddf6891 (informational - the pin wins)
repo pinned OK


In [32]:
for rel in MANIFESTS:
    attrs = subprocess.run(['git', 'check-attr', 'text', 'eol', '--', rel],
                           capture_output=True, text=True,
                           check=True).stdout
    print(attrs.strip())
    assert 'eol: crlf' in attrs, (
        'STOP: ' + rel + ' is not checked out with the registered CRLF bytes (the dataset locks are MD5s of the CRLF form)')
print('manifest EOL pin OK')

def run(cmd, log=None):
    """Run a repo command; any non-zero exit stops the cell."""
    cmd = [str(c) for c in cmd]
    print('+', ' '.join(cmd), flush=True)
    if log:
        with open(log, 'a', encoding='utf-8') as fh:
            p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in p.stdout:
                print(line, end='', flush=True)
                fh.write(line)
            rc = p.wait()
        if rc:
            raise SystemExit('STOP rc=' + str(rc) + ': ' + ' '.join(cmd) + ' (log: ' + log + ')')
    else:
        rc = subprocess.run(cmd, cwd=REPO).returncode
        if rc:
            raise SystemExit('STOP rc=' + str(rc) + ': ' + ' '.join(cmd))

PY = sys.executable
print('runner ready; PY =', PY)


docs/rejection_experiment/adaptation_set_manifest.json: text: set
docs/rejection_experiment/adaptation_set_manifest.json: eol: crlf
docs/rejection_experiment/fresh_set_manifest.json: text: set
docs/rejection_experiment/fresh_set_manifest.json: eol: crlf
manifest EOL pin OK
runner ready; PY = /usr/bin/python3


## S2 - Drive mount + teacher restore (md5-gated)

One-time manual step: the shipped checkpoint is gitignored, so a fresh runtime does not have it. Upload
`models/checkpoints/wastelens_rej_shipped_best.keras` (md5 `8735019226412c4d67b0a17269dc38ad`, 13,583,639 B) from the laptop repo to `MyDrive/WasteLens/iteration12/checkpoints/` before S5/S6. The cell fails loudly with the exact instructions if it is absent - training never starts from another model.


In [33]:
from google.colab import drive
drive.mount('/content/drive')
BK = '/content/drive/MyDrive/WasteLens/iteration12'
for sub in ('datasets/adaptation_set', 'datasets/fresh_set',
            'checkpoints', 'evidence', 'logs'):
    os.makedirs(BK + '/' + sub, exist_ok=True)
print('backup root:', BK)
run([PY, 'src/restore_teacher_ckpt.py', '--drive', BK])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
backup root: /content/drive/MyDrive/WasteLens/iteration12
+ /usr/bin/python3 src/restore_teacher_ckpt.py --drive /content/drive/MyDrive/WasteLens/iteration12


## S3 - materialize datasets (thumb_url only, byte+sha locked, bounded retries)


In [42]:
for manifest, out, drive_sub, report in (
        (MANIFESTS[0], 'scratch/adaptation_set',
         'datasets/adaptation_set', 'adaptation_lock_report.json'),
        (MANIFESTS[1], 'scratch/fresh_set',
         'datasets/fresh_set', 'fresh_lock_report.json')):
    run([PY, '-u', 'src/materialize_manifest.py',
         '--manifest', manifest, '--out', out,
         '--drive', BK + '/' + drive_sub,
         '--report', BK + '/datasets/' + report])
print('datasets materialized: 94/94 + 131/131 or this cell stopped')


+ /usr/bin/python3 -u src/materialize_manifest.py --manifest docs/rejection_experiment/adaptation_set_manifest.json --out scratch/adaptation_set --drive /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_set --report /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json
+ /usr/bin/python3 -u src/materialize_manifest.py --manifest docs/rejection_experiment/fresh_set_manifest.json --out scratch/fresh_set --drive /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_set --report /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_lock_report.json


SystemExit: STOP rc=1: /usr/bin/python3 -u src/materialize_manifest.py --manifest docs/rejection_experiment/fresh_set_manifest.json --out scratch/fresh_set --drive /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_set --report /content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_lock_report.json

In [43]:
import json

report = "/content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_lock_report.json"

with open(report, "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

{
  "generated_utc": "2026-09-30 18:18:27 UTC",
  "manifest": "docs/rejection_experiment/fresh_set_manifest.json",
  "manifest_entries": 131,
  "expected_counts": {
    "supported-organic": 16,
    "supported-recyclable": 21,
    "supported-hazardous": 15,
    "supported-general": 11,
    "ambiguous": 10,
    "ood-objects": 15,
    "ood-clothes": 8,
    "ood-shoes": 8,
    "ood-scenes": 12,
    "multi-object": 15
  },
  "out_dir": "scratch/fresh_set",
  "drive_dir": "/content/drive/MyDrive/WasteLens/iteration12/datasets/fresh_set",
  "verified": 120,
  "resumed_local": 120,
  "restored_from_drive": 0,
  "downloaded": 0,
  "missing": [],
  "sha_mismatch": [
    {
      "file": "supported-organic__Compost__05.jpg",
      "attempts": 4,
      "got_bytes": 493785,
      "want_bytes": 493837,
      "got_sha256": "2c2345225280ca9129cd120a4758dd47e8e9f013d8900f4dff5b0e18352940e3",
      "want_sha256": "159d2456ce057eb0091da6529524f716cc37c06ca73438717be813972abd36be"
    },
    {
      "file"

In [36]:
import subprocess

cmd = [
    "python", "-u",
    "src/materialize_manifest.py",
    "--manifest", "docs/rejection_experiment/adaptation_set_manifest.json",
    "--out", "scratch/adaptation_set",
    "--drive", "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_set",
    "--report", "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json",
]

print("Running materializer directly...\n")

result = subprocess.run(
    cmd,
    cwd="/content/WasteLens",
    text=True,
    capture_output=True
)

print("===== STDOUT =====")
print(result.stdout)

print("===== STDERR =====")
print(result.stderr)

print("===== RETURN CODE =====")
print(result.returncode)

Running materializer directly...

===== STDOUT =====
materialize adaptation_set_manifest.json: 89/94 verified, 0 missing, 5 mismatched; report -> /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json

===== STDERR =====
INCOMPLETE: 0 missing, 5 mismatched - see /content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json

===== RETURN CODE =====
1


In [38]:
import json

report = "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_lock_report.json"

with open(report, "r", encoding="utf-8") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

{
  "generated_utc": "2026-09-30 18:01:27 UTC",
  "manifest": "docs/rejection_experiment/adaptation_set_manifest.json",
  "manifest_entries": 94,
  "expected_counts": {
    "adapt-supported": 60,
    "adapt-ood": 28,
    "adapt-multi": 6
  },
  "out_dir": "scratch/adaptation_set",
  "drive_dir": "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_set",
  "verified": 89,
  "resumed_local": 89,
  "restored_from_drive": 0,
  "downloaded": 0,
  "missing": [],
  "sha_mismatch": [
    {
      "file": "adapt-supported__Button_cells__04.png",
      "attempts": 4,
      "got_bytes": 1446695,
      "want_bytes": 1446695,
      "got_sha256": "b11d136b87bed575e2081b2631082b9c0ee6df0c9b6ea84b93813e164d9e081a",
      "want_sha256": "efcec50933f1144aa79b2a12cd9618373d22cbffdb56a3fb1258bf6f3dc9e757"
    },
    {
      "file": "adapt-supported__Button_cells__05.png",
      "attempts": 4,
      "got_bytes": 725199,
      "want_bytes": 725199,
      "got_sha256": "12794808be07725f6b83e6460a

In [41]:
from pathlib import Path

drive_dir = Path(
    "/content/drive/MyDrive/WasteLens/iteration12/datasets/adaptation_set"
)

files = [
    "adapt-supported__Button_cells__04.png",
    "adapt-supported__Button_cells__05.png",
    "adapt-supported__Litter__07.jpg",
    "adapt-supported__Paper_towels__06.jpg",
    "adapt-ood__Shoes__03.jpg",
]

for name in files:
    p = drive_dir / name
    print(name, "→", "FOUND ✅" if p.exists() else "MISSING ❌")

adapt-supported__Button_cells__04.png → FOUND ✅
adapt-supported__Button_cells__05.png → FOUND ✅
adapt-supported__Litter__07.jpg → FOUND ✅
adapt-supported__Paper_towels__06.jpg → FOUND ✅
adapt-ood__Shoes__03.jpg → FOUND ✅


## S4 - strict preflight (manifest locks + 94/94 + 131/131 + teacher md5)


In [ ]:
run([PY, '-u', 'src/iter12_preflight.py',
     '--report', BK + '/datasets/head12_preflight.json'])


## S5 - train seed 42 (6 epochs, FINAL epoch = candidate) + back up immediately


In [ ]:
run([PY, '-u', 'src/train_distill_adaptation.py', '--seed', '42'],
    log=BK + '/logs/head12s42_train.log')
run([PY, 'src/backup_iter12.py', 'save', '--drive', BK, '--seed', '42'])
import glob
print('seed-42 checkpoints:', sorted(glob.glob(
    'models/checkpoints/wastelens_rej_head12s42*')))
print('seed-42 evidence docs:', sorted(glob.glob(
    'docs/rejection_experiment/head12s42*')))


## S6 - train seed 43 (identical recipe) + back up immediately


In [ ]:
run([PY, '-u', 'src/train_distill_adaptation.py', '--seed', '43'],
    log=BK + '/logs/head12s43_train.log')
run([PY, 'src/backup_iter12.py', 'save', '--drive', BK, '--seed', '43'])
print('seed-43 checkpoints:', sorted(glob.glob(
    'models/checkpoints/wastelens_rej_head12s43*')))
print('seed-43 evidence docs:', sorted(glob.glob(
    'docs/rejection_experiment/head12s43*')))


## S7 - evaluate both seeds (frozen threshold 0.0702, head12 gate table) + summary


In [ ]:
for seed in ('42', '43'):
    run([PY, '-u', 'src/eval_head_adaptation.py', '--seed', seed,
         '--tag', 'head12', '--protocol',
         'iteration12_distillation_protocol.md',
         '--gates', 'head12_gates.json'],
        log=BK + '/logs/head12s' + seed + '_eval.log')
run([PY, '-u', 'src/eval_head_adaptation.py', '--summary',
     '--tag', 'head12', '--protocol',
     'iteration12_distillation_protocol.md'],
    log=BK + '/logs/head12_summary.log')
run([PY, 'src/backup_iter12.py', 'save', '--drive', BK])
print('eval + decision mirrored; production model/threshold untouched')


## S8 - strict evidence bundle (refuses to build if anything is missing)


In [ ]:
run([PY, 'src/package_head12_evidence.py', '--drive',
     BK + '/evidence', '--out', 'head12_evidence.tar.gz'])
import os
for p in ('head12_evidence.tar.gz',
          BK + '/evidence/head12_evidence.tar.gz'):
    print(p, os.path.getsize(p), 'bytes')


## S9 - SAVE NOW (run any time before the runtime times out)


In [ ]:
run([PY, 'src/backup_iter12.py', 'save', '--drive', BK])
print('full artifact set re-mirrored to', BK)


## S10 - RESTORE from Drive (fresh runtime) + disaster recovery

1. run S0, S1 (pin), S2 (Drive + teacher restore);
2. run this cell - it restores every finished artifact from Drive and re-materializes the datasets (verified files are reused, never re-downloaded);
3. run the S11 status cell - it lists exactly what exists;
4. resume at the FIRST missing step only (S5/S6 for a seed without _final.keras, then S7, S8). Never re-run a completed training cell;
5. nothing on Drive yet? this cell stops with a clear message - start from S3 instead.


In [ ]:
run([PY, 'src/backup_iter12.py', 'restore', '--drive', BK])
for manifest, out, drive_sub, report in (
        (MANIFESTS[0], 'scratch/adaptation_set',
         'datasets/adaptation_set', 'adaptation_lock_report.json'),
        (MANIFESTS[1], 'scratch/fresh_set',
         'datasets/fresh_set', 'fresh_lock_report.json')):
    run([PY, '-u', 'src/materialize_manifest.py',
         '--manifest', manifest, '--out', out,
         '--drive', BK + '/' + drive_sub,
         '--report', BK + '/datasets/' + report])
print('restore complete - run the S11 status cell for the resume point')


## S11 - DISASTER RECOVERY STATUS (pure python)


In [ ]:
import glob, hashlib, os, subprocess

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def n_files(d):
    if not os.path.isdir(d):
        return 0
    return len([f for f in os.listdir(d)
                if os.path.isfile(os.path.join(d, f))])

head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO,
                      capture_output=True, text=True).stdout.strip()
print('HEAD               :', head)
print('pinned as expected :', head == PINNED)
print('git status --short :')
print(subprocess.run(['git', 'status', '--short'], cwd=REPO,
                     capture_output=True, text=True).stdout.strip()
      or '  (clean)')
print('adaptation_set     :', n_files('scratch/adaptation_set'), '/ 94')
print('fresh_set          :', n_files('scratch/fresh_set'), '/ 131')
for rel in MANIFESTS:
    print('manifest md5       :', md5(rel), '<-', rel)
teacher = 'models/checkpoints/wastelens_rej_shipped_best.keras'
print('teacher ckpt md5   :',
      md5(teacher) if os.path.exists(teacher) else 'MISSING',
      '(want 8735019226412c4d67b0a17269dc38ad)')
print('threshold          : 0.0702 (frozen; production untouched)')
print('seed-42 ckpts      :', sorted(os.path.basename(p) for p in
      glob.glob('models/checkpoints/wastelens_rej_head12s42*')))
print('seed-43 ckpts      :', sorted(os.path.basename(p) for p in
      glob.glob('models/checkpoints/wastelens_rej_head12s43*')))
print('seed-42 docs       :', sorted(os.path.basename(p) for p in
      glob.glob('docs/rejection_experiment/head12s42*')))
print('seed-43 docs       :', sorted(os.path.basename(p) for p in
      glob.glob('docs/rejection_experiment/head12s43*')))
print('decision/evidence  :', [p for p in (
    'docs/rejection_experiment/iteration12_decision.md',
    'docs/rejection_experiment/head12_seed_consistency.json',
    'head12_evidence.tar.gz',
    BK + '/evidence/head12_evidence.tar.gz') if os.path.exists(p)])
print('drive root listing :',
      sorted(os.listdir(BK)) if os.path.isdir(BK) else 'MISSING')
print()
print('RESUME RULE: start at S5/S6 for any seed without a _final.keras, then S7 and S8; never re-run a completed training cell.')


## Locks and safety rails (quick reference)

* commit pin: this notebook runs `git checkout --force <PINNED>` and asserts HEAD == PINNED; PINNED is the commit that carries the fixed pipeline.
* dataset manifests: md5 `acae0933aca123c835c1e55396af182a` (94 files) and `99540f56efb95e2c26b2b6cc9414ff5d` (131 images); CRLF checkout pinned by `.gitattributes` so Linux reproduces the registered bytes; the trainer re-checks the lock itself.
* teacher: md5 `8735019226412c4d67b0a17269dc38ad`, restored from Drive and verified before S5/S6 may run.
* threshold: 0.0702 (frozen); the production model is untouched by every cell here.
* every heavy cell stops the run on a non-zero exit; nothing is silently skipped.
